In [0]:
# =============================================================================
# NYC Taxi Medallion Architecture Pipeline - Test Notebook
# =============================================================================
# This notebook orchestrates and tests the complete end-to-end pipeline:
# RAW (Volume) → BRONZE (Delta) → SILVER (Delta) → GOLD (Delta)
#
# Project Structure:
#   src/config.py          - Pipeline configuration
#   src/setup.py           - Unity Catalog setup
#   src/bronze/download.py - Raw data download
#   src/bronze/ingest.py   - Bronze ingestion
#   src/silver/transform.py - Silver transformation
#   src/gold/create_tables.py - Gold table creation
# =============================================================================

# Enable autoreload - automatically reload modules when they change
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/Workspace/Users/jj4evers2s2@gmail.com/nyc-taxi-medallion-pipeline/src")

# Import pipeline modules
from config import (
    BASE_URL, CATALOG, RAW_SCHEMA, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA,
    VOLUME, YEARS, MONTHS, TAXI_TYPES, RAW_PATH, print_config
)
from setup import setup_catalog
from bronze.download import download_all_parallel
from bronze.ingest import ingest_to_bronze, ingest_all_taxi_types
from silver.transform import transform_to_silver, transform_all_taxi_types
from gold.create_tables import create_gold_table, create_all_gold_tables

print("Pipeline modules imported successfully\n")
print_config()

Pipeline modules imported successfully

NYC Taxi Pipeline Configuration
  Catalog: nyc_taxi
  Raw path: /Volumes/nyc_taxi/raw/files
  Processing: ['yellow', 'green', 'fhv', 'fhvhv']
  Period: [2023], months [1, 2, 3, 4, 5]


In [0]:
# =============================================================================
# STEP 1: Unity Catalog Setup
# =============================================================================
# Creates the complete medallion architecture in Unity Catalog:
# - Catalog: nyc_taxi
# - Schemas: raw, bronze, silver, gold (with governance metadata)
# - Volume: raw/files (for Parquet storage)
# =============================================================================

setup_catalog(spark)

print("\n✓ Unity Catalog setup completed successfully!")


  Initializing medallion architecture setup...

Catalog created: nyc_taxi
Schema RAW: nyc_taxi.raw
Volume: /Volumes/nyc_taxi/raw/files/
Schema BRONZE: nyc_taxi.bronze
Schema SILVER: nyc_taxi.silver
Schema GOLD: nyc_taxi.gold

 Bronze tables to be created:
  - nyc_taxi.bronze.yellow_trips
  - nyc_taxi.bronze.green_trips
  - nyc_taxi.bronze.fhv_trips
  - nyc_taxi.bronze.fhvhv_trips

Setup completed successfully!

✓ Unity Catalog setup completed successfully!


In [0]:
# =============================================================================
# STEP 2: Download Raw Data from NYC Open Data
# =============================================================================
# Downloads Parquet files from NYC TLC for all configured taxi types,
# years, and months. Features:
# - Parallel downloads (4 workers)
# - Retry logic with exponential backoff (3 retries)
# - Idempotency (skips existing valid files)
# - Comprehensive error handling and reporting
# =============================================================================

stats = download_all_parallel(BASE_URL, RAW_PATH, TAXI_TYPES, YEARS, MONTHS)

print(f"\n✓ Download completed: {stats['downloaded']} new, {stats['skipped']} skipped, {stats['failed']} failed")
print(f"  Total data: {stats['total_mb']:.1f} MB")

Starting parallel download: 20 files, 4 workers

Download complete: 0 new, 20 skipped, 0 failed
Total: 2801.3 MB in 2.8s

✓ Download completed: 0 new, 20 skipped, 0 failed
  Total data: 2801.3 MB


In [0]:
# =============================================================================
# STEP 3: Bronze Layer Ingestion
# =============================================================================
# Ingests raw Parquet files into Bronze Delta tables for all taxi types.
# Features:
# - Schema normalization (lowercase columns)
# - Audit metadata (_source_file, _ingestion_timestamp)
# - Schema evolution support (mergeSchema)
# - Automatic optimization (OPTIMIZE)
# - Union of files with missing column handling
# =============================================================================

bronze_results = ingest_all_taxi_types(
    spark=spark,
    catalog=CATALOG,
    raw_path=RAW_PATH,
    bronze_schema=BRONZE_SCHEMA,
    taxi_types=TAXI_TYPES,
    years=YEARS,      # Only process configured years
    months=MONTHS,    # Only process configured months (respects config.py)
    overwrite=True    # Set False for incremental ingestion
)

print("\n✓ Bronze ingestion completed for all taxi types!")


Processing: YELLOW
Found 5 parquet file(s) matching configured period
  Years: [2023], Months: [1, 2, 3, 4, 5]
  (Skipped 4 files outside configured period)
Success: 16,186,386 records in 11.7s
Table: nyc_taxi.bronze.yellow_trips

Processing: GREEN
Found 5 parquet file(s) matching configured period
  Years: [2023], Months: [1, 2, 3, 4, 5]
  (Skipped 4 files outside configured period)
Success: 339,630 records in 9.9s
Table: nyc_taxi.bronze.green_trips

Processing: FHV
Found 5 parquet file(s) matching configured period
  Years: [2023], Months: [1, 2, 3, 4, 5]
  (Skipped 4 files outside configured period)
Success: 6,185,664 records in 12.4s
Table: nyc_taxi.bronze.fhv_trips

Processing: FHVHV
Found 5 parquet file(s) matching configured period
  Years: [2023], Months: [1, 2, 3, 4, 5]
  (Skipped 4 files outside configured period)
Success: 95,846,120 records in 31.9s
Table: nyc_taxi.bronze.fhvhv_trips


BRONZE INGESTION SUMMARY
  YELLOW  : 16,186,386 records (11.72s)
  GREEN   : 339,630 reco

In [0]:
# =============================================================================
# STEP 4: Silver Layer Transformation
# =============================================================================
# Transforms Bronze data into clean, validated Silver tables for all taxi types.
# Quality rules applied:
# - Timestamp validation (non-null, pickup before dropoff)
# - Temporal outlier removal (volumetric filter for corrupted years)
# - Distance validation (0-500 miles for yellow/green)
# - Fare validation (non-negative for yellow/green)
# - Passenger count validation (1-6 for yellow/green)
# - Duplicate removal based on business keys
# - Derived metrics (trip_duration_minutes, pickup_date)
# =============================================================================

silver_results = transform_all_taxi_types(
    spark=spark,
    catalog=CATALOG,
    bronze_schema=BRONZE_SCHEMA,
    silver_schema=SILVER_SCHEMA,
    taxi_types=TAXI_TYPES
)

print("\nSilver transformation completed for all taxi types!")


Transforming: YELLOW (Bronze to Silver)

  Writing Silver table with Liquid Clustering...
  Silver records: 15,060,826

✓ Success: 15,060,826 records in 24.1s
  Table: nyc_taxi.silver.yellow_trips_clean
  Clustering: pickup_date (Liquid)
  Throughput: 625,274 records/sec

Transforming: GREEN (Bronze to Silver)

  Writing Silver table with Liquid Clustering...
  Silver records: 296,501

✓ Success: 296,501 records in 6.2s
  Table: nyc_taxi.silver.green_trips_clean
  Clustering: pickup_date (Liquid)
  Throughput: 47,846 records/sec

Transforming: FHV (Bronze to Silver)

  Writing Silver table with Liquid Clustering...
  Silver records: 6,005,471

✓ Success: 6,005,471 records in 13.3s
  Table: nyc_taxi.silver.fhv_trips_clean
  Clustering: pickup_date (Liquid)
  Throughput: 451,089 records/sec

Transforming: FHVHV (Bronze to Silver)

  Writing Silver table with Liquid Clustering...
  Silver records: 95,268,489

✓ Success: 95,268,489 records in 100.7s
  Table: nyc_taxi.silver.fhvhv_trips_cl

In [0]:
# =============================================================================
# STEP 5: Gold Layer - Consumption-Ready Tables
# =============================================================================
# Creates Gold consumption tables optimized for analytics.
# Features:
# - Column standardization (unified naming: pickup_datetime, dropoff_datetime)
# - Temporal dimensions (hour, day_of_week, is_weekend, year_month)
# - Business metrics (avg_speed_mph)
# - Query optimization (partitioned by year_month, z-ordered by location IDs)
# - Comprehensive table and column documentation
#
# Note: Only supports yellow and green taxis (FHV/FHVHV lack required columns)
# =============================================================================

gold_results = create_all_gold_tables(
    spark=spark,
    catalog=CATALOG,
    silver_schema=SILVER_SCHEMA,
    gold_schema=GOLD_SCHEMA,
    taxi_types=["yellow", "green"]  # Only yellow/green have required consumption columns
)

print("\n✓ Gold layer created successfully!")


Creating Gold Table: YELLOW

  Writing Gold table with Liquid Clustering...
  Gold records: 15,060,826

✓ Success: 15,060,826 records in 20.8s
  Table: nyc_taxi.gold.yellow_trips_gold
  Clustering: pickup_date (Liquid)

Creating Gold Table: GREEN

  Writing Gold table with Liquid Clustering...
  Gold records: 296,501

✓ Success: 296,501 records in 12.2s
  Table: nyc_taxi.gold.green_trips_gold
  Clustering: pickup_date (Liquid)


GOLD LAYER CREATION SUMMARY
  YELLOW  : 15,060,826 records (20.76s)
   Table: nyc_taxi.gold.yellow_trips_gold
  GREEN   : 296,501 records (12.16s)
   Table: nyc_taxi.gold.green_trips_gold

Total Gold records: 15,357,327
✓ Liquid clustering enabled (pickup_date) - consistent across layers

✓ Gold layer created successfully!


In [0]:
# =============================================================================
# DATA QUALITY VALIDATION - GOLD LAYER
# =============================================================================
# Comprehensive validation of Gold layer:
# - Schema validation (required columns)
# - Data quality checks (nulls, completeness)
# - Business metrics summary
# - Temporal distribution analysis
# - Sample data preview
# =============================================================================

from pyspark.sql import functions as F

taxi_type_to_validate = "yellow"  # Change to "green" to validate green taxi
gold_table = f"{CATALOG}.{GOLD_SCHEMA}.{taxi_type_to_validate}_trips_gold"

print(f"\nGOLD LAYER VALIDATION - {taxi_type_to_validate.upper()}")
print("="*60)

try:
    df_gold = spark.table(gold_table)
    
    # 1. Schema validation - Check required columns
    print("\n1. Schema Validation:")
    required_cols = [
        "vendor_id", "passenger_count", "total_amount",
        "pickup_datetime", "dropoff_datetime"
    ]
    
    missing_cols = [col for col in required_cols if col not in df_gold.columns]
    
    if missing_cols:
        print(f"   Missing required columns: {missing_cols}")
    else:
        print(f"   All required consumption columns present")
    
    print(f"   Total columns: {len(df_gold.columns)}")
    print(f"   Column list: {', '.join(df_gold.columns[:10])}...")
    
    # 2. Data quality checks
    print("\n2. Data Quality:")
    total_records = df_gold.count()
    print(f"   Total records: {total_records:,}")
    
    # Check for nulls in critical columns
    null_checks = df_gold.select([
        (F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) / total_records * 100).alias(f"{c}_null_pct")
        for c in required_cols
    ])
    
    print("\n   Null % in required columns:")
    null_checks.display()
    
    # 3. Business metrics summary
    print("\n3. Business Metrics:")
    df_gold.select(
        F.count("*").alias("total_trips"),
        F.round(F.avg("passenger_count"), 2).alias("avg_passengers"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
        F.round(F.avg("avg_speed_mph"), 2).alias("avg_speed_mph")
    ).display()
    
    # 4. Temporal distribution
    print("\n4. Temporal Distribution:")
    df_gold.groupBy("pickup_date") \
        .agg(F.count("*").alias("trips")) \
        .orderBy("pickup_date") \
        .display()
    
    # 5. Weekend vs Weekday
    print("\n5. Weekend vs Weekday Analysis:")
    df_gold.groupBy("is_weekend") \
        .agg(
            F.count("*").alias("trips"),
            F.round(F.avg("total_amount"), 2).alias("avg_fare"),
            F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration")
        ) \
        .display()
    
    # 6. Sample data preview
    print("\n6. Sample Data (first 10 rows):")
    df_gold.select(
        "vendor_id", "passenger_count", "total_amount",
        "pickup_datetime", "pickup_hour", "is_weekend",
        "trip_duration_minutes", "avg_speed_mph"
    ).limit(10).display()
    
    print("\nGold layer validation completed successfully!")
    
except Exception as e:
    print(f"\nValidation failed: {str(e)}")
    print("\nMake sure to execute the Gold creation cell first.")


GOLD LAYER VALIDATION - YELLOW

1. Schema Validation:
   All required consumption columns present
   Total columns: 18
   Column list: vendor_id, passenger_count, total_amount, pickup_datetime, dropoff_datetime, trip_duration_minutes, trip_distance, pickup_date, pickup_location_id, dropoff_location_id...

2. Data Quality:
   Total records: 15,060,826

   Null % in required columns:


vendor_id_null_pct,passenger_count_null_pct,total_amount_null_pct,pickup_datetime_null_pct,dropoff_datetime_null_pct
0.0,0.0,0.0,0.0,0.0



3. Business Metrics:


total_trips,avg_passengers,avg_total_amount,avg_duration_min,avg_speed_mph
15060826,1.39,28.22,16.91,12.46



4. Temporal Distribution:


...


5. Weekend vs Weekday Analysis:


is_weekend,trips,avg_fare,avg_duration
true,4126095,27.58,16.23
false,10934731,28.46,17.17



6. Sample Data (first 10 rows):


vendor_id,passenger_count,total_amount,pickup_datetime,pickup_hour,is_weekend,trip_duration_minutes,avg_speed_mph
2,1.0,17.16,2023-05-23T00:25:06.000,0,false,7.566666666666666,8.881057268722467
2,1.0,17.1,2023-05-23T00:15:54.000,0,false,1424.2833333333333,0.07330002223340394
2,1.0,26.15,2023-05-23T00:07:49.000,0,false,14.216666666666667,21.566236811254395
2,1.0,38.75,2023-05-23T01:49:29.000,1,false,19.566666666666666,25.635434412265756
2,1.0,8.7,2023-05-23T01:47:16.000,1,false,1.0833333333333333,3.876923076923078
2,1.0,61.85,2023-05-23T01:01:01.000,1,false,28.683333333333334,30.352120859965137
2,1.0,13.49,2023-05-23T03:42:38.000,3,false,7.216666666666667,11.556581986143186
2,1.0,55.85,2023-05-23T03:29:15.000,3,false,26.9,23.576208178438662
2,3.0,30.6,2023-05-23T03:02:50.000,3,false,17.233333333333334,14.065764023210832
2,1.0,12.9,2023-05-23T05:47:18.000,5,false,5.7,11.052631578947368



Gold layer validation completed successfully!


In [0]:
# =============================================================================
# DATA QUALITY VALIDATION - SILVER LAYER
# =============================================================================
# Quality analysis of Silver layer:
# - Trip statistics (averages, totals)
# - Distribution by period
# - Data completeness metrics
# =============================================================================

from pyspark.sql import functions as F

taxi_type_to_analyze = "yellow"  # Change this to analyze different taxi types
silver_table = f"{CATALOG}.{SILVER_SCHEMA}.{taxi_type_to_analyze}_trips_clean"

print(f"\nQUALITY ANALYSIS - SILVER LAYER ({taxi_type_to_analyze.upper()})")
print("="*60)

# Estatísticas descritivas
df_silver = spark.table(silver_table)

print("\n1. Trip Statistics:")

# Generic stats for all taxi types
stats_cols = [
    F.count("*").alias("total_trips"),
    F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_min")
]

# Add fare stats only for yellow/green (they have fare columns)
if taxi_type_to_analyze in ["yellow", "green"]:
    stats_cols.extend([
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount")
    ])

df_silver.select(stats_cols).display()

print("\n2. Distribution by Period:")
df_silver.groupBy("pickup_date") \
    .agg(F.count("*").alias("trips")) \
    .orderBy("pickup_date") \
    .display()

print("\n3. Data Completeness (% non-null):")
total = df_silver.count()

# Select columns that exist in the dataframe
available_cols = df_silver.columns
check_cols = []

# Common columns to check across all taxi types
for col_name in ["trip_duration_minutes", "pickup_date"]:
    if col_name in available_cols:
        check_cols.append(col_name)

# Yellow/Green specific columns
if taxi_type_to_analyze in ["yellow", "green"]:
    for col_name in ["passenger_count", "ratecodeid", "payment_type", "fare_amount"]:
        if col_name in available_cols:
            check_cols.append(col_name)

if check_cols:
    completeness = df_silver.select([
        (F.count(F.when(F.col(c).isNotNull(), c)) / total * 100).alias(c)
        for c in check_cols
    ])
    completeness.display()
else:
    print("No applicable columns found for completeness check")

print("\nQuality analysis completed")


QUALITY ANALYSIS - SILVER LAYER (YELLOW)

1. Trip Statistics:


total_trips,avg_duration_min,avg_distance_miles,avg_fare,avg_total_amount
15060826,16.91,3.47,19.19,28.22



2. Distribution by Period:


pickup_date,trips
2023-01-01,70031
2023-01-02,61905
2023-01-03,80574
2023-01-04,89539
2023-01-05,95208
2023-01-06,96641
2023-01-07,99200
2023-01-08,80185
2023-01-09,80090
2023-01-10,94341



3. Data Completeness (% non-null):


trip_duration_minutes,pickup_date,passenger_count,ratecodeid,payment_type,fare_amount
100.0,100.0,100.0,100.0,100.0,100.0



Quality analysis completed


In [0]:
# =============================================================================
# GOVERNANCE AND METADATA INSPECTION
# =============================================================================
# Explore Unity Catalog structure:
# - List schemas and tables
# - View schema properties and governance metadata
# =============================================================================

print("NYC Taxi Catalog Summary\n")

# 1. List all schemas
print("1. Available schemas:")
spark.sql(f"SHOW SCHEMAS IN {CATALOG}").display()

# 2. List tables by schema
print("\n2. Bronze tables:")
spark.sql(f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}").display()

print("\n3. Silver tables:")
spark.sql(f"SHOW TABLES IN {CATALOG}.{SILVER_SCHEMA}").display()

print("\n4. Gold tables:")
spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}").display()

# 3. Schema properties
print("\n5. Bronze Schema properties:")
spark.sql(f"DESCRIBE SCHEMA EXTENDED {CATALOG}.{BRONZE_SCHEMA}").display()

NYC Taxi Catalog Summary

1. Available schemas:


databaseName
bronze
default
gold
information_schema
raw
silver



2. Bronze tables:


database,tableName,isTemporary
bronze,fhv_trips,false
bronze,fhvhv_trips,false
bronze,green_trips,false
bronze,yellow_trips,false



3. Silver tables:


database,tableName,isTemporary
silver,fhv_trips_clean,false
silver,fhvhv_trips_clean,false
silver,green_trips_clean,false
silver,yellow_trips_clean,false



4. Gold tables:


database,tableName,isTemporary
gold,green_trips_gold,false
gold,yellow_trips_gold,false



5. Bronze Schema properties:


database_description_item,database_description_value
Catalog Name,nyc_taxi
Namespace Name,bronze
Comment,
Collation,UTF8_BINARY
Location,
Owner,jj4evers2s2@gmail.com
Properties,"((description,Ingestion layer - raw data in Delta format with audit metadata), (layer,bronze))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_aws_us_east_2)


In [0]:
# =============================================================================
# DELTA LAKE HISTORY AND STATISTICS
# =============================================================================
# View Delta Lake table metadata:
# - Version history (time travel)
# - Table details and statistics
# - Operation metrics
# =============================================================================

print("Version History - Bronze Yellow\n")

# Version history (Delta Time Travel)
bronze_table = f"{CATALOG}.{BRONZE_SCHEMA}.yellow_trips"

try:
    spark.sql(f"DESCRIBE HISTORY {bronze_table}").select(
        "version", "timestamp", "operation", "operationMetrics"
    ).display()
except:
    print("Table does not exist yet. Execute ingestion cells first.")

print("\nSilver Yellow Table Statistics\n")

silver_table = f"{CATALOG}.{SILVER_SCHEMA}.yellow_trips_clean"

try:
    # Table details
    spark.sql(f"DESCRIBE DETAIL {silver_table}").display()
except:
    print("Table does not exist yet. Execute Silver transformations first.")

Version History - Bronze Yellow



version,timestamp,operation,operationMetrics
2,2026-06-09T18:53:24.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 5, numRemovedFiles -> 9, numRemovedBytes -> 463365900, numDeletionVectorsRemoved -> 0, numOutputRows -> 16186386, numOutputBytes -> 267032528)"
1,2026-06-09T18:44:01.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 9, numRemovedFiles -> 5, numRemovedBytes -> 249324731, numDeletionVectorsRemoved -> 0, numOutputRows -> 28071659, numOutputBytes -> 463365900)"
0,2026-06-06T18:11:40.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 5, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 16186386, numOutputBytes -> 249324731)"



Silver Yellow Table Statistics



format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,6b1e8aef-7c47-4493-805a-876b364f3df8,nyc_taxi.silver.yellow_trips_clean,"Yellow trips - clean data. Rules: valid timestamps (2023-01-01 to 2023-05-31), distance/fare/passenger validated. Liquid clustered by pickup_date.",,2026-06-07T00:16:50.789Z,2026-06-09T20:30:40.000Z,List(),List(pickup_date),4,278973214,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, timestampNtz)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
